# MotorCam – fáze 1: Data (Google Colab)

Notebook stáhne a převede veřejné datasety. Data se ukládají na **Google Drive**,
aby po odpojení Colabu nezmizela a nemusela se stahovat znovu.

1. Připojení Drive a stažení repozitáře
2. RDD2022 → YOLO (díry, trhliny)
3. RSCD → 5 tříd povrchu
4. Kontrola převodu (ukázky s rámečky) a grafy do práce

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/motorcam'   # sem se uloží data a výsledky
REPO_URL = 'https://github.com/MiniGue1/motorcam.git'  # u soukromého repa: https://<token>@github.com/...

In [ ]:
import os
%cd /content
if not os.path.exists('motorcam'):
    !git clone -q $REPO_URL motorcam
%cd /content/motorcam
!pip install -q -r requirements.txt

# Na Google Drive ukládáme jen stažené ARCHIVY a VÝSLEDKY (přežijí odpojení Colabu).
# Rozbalená a převedená data (desítky tisíc souborů) držíme na rychlém lokálním disku –
# zápis tolika souborů na Drive by trval velmi dlouho. Po odpojení se jen znovu rozbalí.
for d in ['data/raw/rdd2022/archives', 'data/raw/rscd/archives', 'data/custom', 'results']:
    target = f'{DRIVE_DIR}/{d}'
    os.makedirs(target, exist_ok=True)
    os.makedirs(os.path.dirname(d), exist_ok=True)
    if not os.path.islink(d):
        !rm -rf $d
        os.symlink(target, d)
!ls -la data data/raw/rdd2022

## 2. RDD2022

Výchozí země: Czech, Japan, India, United_States, China_MotorBike (≈ 2,4 GB).
Archivy se na Drive ukládají průběžně (a příště se už nestahují) – když Colab spadne, stačí buňku spustit znovu.

In [ ]:
!python -m src.dataset.download rdd2022

In [ ]:
# Převod do YOLO (lokální disk Colabu)
!python -m src.dataset.rdd_to_yolo
!cat data/processed/rdd2022_yolo/data.yaml
!cat results/tables/rdd2022_statistiky.csv

### Kontrola převodu
Náhodné snímky s nakreslenými rámečky – **díra červeně, trhlina oranžově**.
Pokud rámečky nesedí na poškození, je chyba v převodu.

In [ ]:
import random, cv2, matplotlib.pyplot as plt
from pathlib import Path

root = Path('data/processed/rdd2022_yolo')
colors = {0: (227, 73, 72), 1: (237, 161, 0)}  # RGB
imgs = [p for p in (root / 'images/train').glob('*.jpg')
        if (root / 'labels/train' / f'{p.stem}.txt').read_text().strip()]
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, p in zip(axes.flat, random.sample(imgs, 6)):
    im = cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB)
    h, w = im.shape[:2]
    for ln in (root / 'labels/train' / f'{p.stem}.txt').read_text().splitlines():
        c, xc, yc, bw, bh = map(float, ln.split())
        x1, y1 = int((xc - bw / 2) * w), int((yc - bh / 2) * h)
        x2, y2 = int((xc + bw / 2) * w), int((yc + bh / 2) * h)
        cv2.rectangle(im, (x1, y1), (x2, y2), colors[int(c)], 3)
    ax.imshow(im); ax.set_title(p.name); ax.axis('off')
plt.tight_layout()

## 3. RSCD

Odkaz na archiv najdeš na https://thu-rsxd.com/rscd/ (Google Drive).
⚠️ Pokud má archiv víc než 5 GB, zvaž menší verzi – nám stačí ~6 000 snímků na třídu.

In [ ]:
RSCD_URL = ''   # sem vlož odkaz
if RSCD_URL:
    !python -m src.dataset.download rscd --url "$RSCD_URL"
    !python -m src.dataset.rscd_to_classes
    !cat results/tables/rscd_statistiky.csv

In [ ]:
# Ukázka: 4 snímky z každé třídy povrchu
from pathlib import Path
import matplotlib.pyplot as plt, cv2, random
root = Path('data/processed/rscd_cls/train')
classes = sorted(p.name for p in root.iterdir())
fig, axes = plt.subplots(len(classes), 4, figsize=(10, 2.2 * len(classes)))
for r, c in enumerate(classes):
    files = random.sample(list((root / c).iterdir()), 4)
    for k, f in enumerate(files):
        axes[r, k].imshow(cv2.cvtColor(cv2.imread(str(f)), cv2.COLOR_BGR2RGB)); axes[r, k].axis('off')
    axes[r, 0].set_title(c, loc='left')
plt.tight_layout()

## 4. Grafy do práce
Uloží se do `results/figures` (PNG + PDF) a `results/tables` (CSV) – na Drive.

In [ ]:
!python -m src.dataset.dataset_stats
from IPython.display import Image, display
for f in sorted(Path('results/figures').glob('data_*.png')):
    display(Image(str(f)))